# 🏗️ Single-Image 3D Reconstruction — Interactive Demo

This notebook walks through the full pipeline:
1. **Dataset** — Explore the simulated ShapeNet dataset
2. **Model** — Build and inspect the reconstruction network
3. **Training** — Train for a few epochs
4. **Evaluation** — Compute IoU, Chamfer Distance, and completeness
5. **Visualization** — Compare predicted vs ground truth 3D point clouds
6. **Explainability** — SHAP/gradient attribution maps

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. 📊 Explore the Dataset

The simulated ShapeNet dataset procedurally generates paired (RGB image, 3D point cloud) samples for 5 shape categories.

In [ ]:
from src.datasets.shapenet_simulated import SimulatedShapeNetDataset

dataset = SimulatedShapeNetDataset(
    num_samples=500,
    num_points=512,
    image_size=128,
    categories=['cube', 'sphere', 'cylinder', 'cone', 'torus'],
    seed=42,
)

print(f'Dataset size: {len(dataset)}')
print(f'Categories: {dataset.get_categories()}')
print(f'Category counts: {dataset.get_category_counts()}')

In [ ]:
# Visualize samples from each category
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

category_samples = {}
for i in range(len(dataset)):
    s = dataset[i]
    cat = s['category_name']
    if cat not in category_samples:
        category_samples[cat] = s
    if len(category_samples) == 5:
        break

colors = {'cube': '#2878D2', 'sphere': '#DC503C', 'cylinder': '#3CBE5A',
          'cone': '#F0B428', 'torus': '#B43CC8'}

for idx, (cat, sample) in enumerate(category_samples.items()):
    # Image
    axes[0, idx].imshow(sample['image'].permute(1, 2, 0).numpy())
    axes[0, idx].set_title(f'{cat.upper()}', fontsize=13, fontweight='bold')
    axes[0, idx].axis('off')
    
    # Point cloud (top-down view)
    pc = sample['point_cloud'].numpy()
    axes[1, idx].scatter(pc[:, 0], pc[:, 1], c=colors[cat], s=1, alpha=0.6)
    axes[1, idx].set_xlim(-0.6, 0.6)
    axes[1, idx].set_ylim(-0.6, 0.6)
    axes[1, idx].set_aspect('equal')
    axes[1, idx].set_title('Point Cloud (top)', fontsize=10)
    axes[1, idx].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Rendered Image', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('GT Point Cloud', fontsize=12, fontweight='bold')
plt.suptitle('Simulated ShapeNet Dataset — All 5 Categories', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 🧠 Build the Model

The model uses a ResNet-18 encoder to extract image features, followed by an MLP decoder that predicts 3D point coordinates.

In [ ]:
from src.models.reconstruction_net import SingleImageReconstructionNet

model = SingleImageReconstructionNet(
    encoder_cfg={'backbone': 'resnet18', 'pretrained': False, 'latent_dim': 128},
    decoder_cfg={'hidden_dims': [128, 128], 'num_points': 512, 'activation': 'relu'},
)

param_counts = model.get_num_params()
print(f"Total parameters: {param_counts['total']:,}")
print(f"  Encoder: {param_counts['encoder']:,}")
print(f"  Decoder: {param_counts['decoder']:,}")

# Test forward pass
sample = dataset[0]
with torch.no_grad():
    model.eval()
    pred = model(sample['image'].unsqueeze(0))
    print(f"\nInput shape:  {sample['image'].shape}")
    print(f"Output shape: {pred.shape}")
    print(f"Output range: [{pred.min():.3f}, {pred.max():.3f}]")

## 3. 🎯 Quick Training

Train the model for a few epochs to see the loss converge.

In [ ]:
from src.datasets.transforms import ImageTransforms
from src.models.losses import ChamferDistanceLoss
from torch.utils.data import DataLoader

# Datasets with normalization
img_transform = ImageTransforms(color_jitter=False, random_crop=False, normalize=True, image_size=128)

train_ds = SimulatedShapeNetDataset(
    num_samples=300, num_points=512, image_size=128,
    categories=['cube', 'sphere', 'cylinder', 'cone', 'torus'],
    transform=img_transform, seed=42,
)
val_ds = SimulatedShapeNetDataset(
    num_samples=100, num_points=512, image_size=128,
    categories=['cube', 'sphere', 'cylinder', 'cone', 'torus'],
    transform=img_transform, seed=99999,
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

# Model, loss, optimizer
model = SingleImageReconstructionNet(
    encoder_cfg={'backbone': 'resnet18', 'pretrained': False, 'latent_dim': 128},
    decoder_cfg={'hidden_dims': [128, 128], 'num_points': 512},
).to(device)

criterion = ChamferDistanceLoss(reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Training loop
num_epochs = 5
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Train
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        images = batch['image'].to(device)
        gt_points = batch['point_cloud'].to(device)
        
        pred = model(images)
        loss = criterion(pred, gt_points)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            gt_points = batch['point_cloud'].to(device)
            pred = model(images)
            val_loss += criterion(pred, gt_points).item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    print(f'Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.5f} | Val: {val_loss:.5f}')

# Plot loss curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, num_epochs+1), train_losses, 'b-o', label='Train Loss')
ax.plot(range(1, num_epochs+1), val_losses, 'r-o', label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Chamfer Distance')
ax.set_title('Training Progress', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 📈 Evaluation

Compute IoU, Chamfer Distance, and reconstruction completeness on the validation set.

In [ ]:
from src.evaluation.metrics import compute_chamfer_distance, compute_iou, compute_reconstruction_completeness

model.eval()
all_cd, all_iou, all_comp = [], [], []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        gt = batch['point_cloud'].to(device)
        pred = model(images)
        
        cd = compute_chamfer_distance(pred, gt)
        iou = compute_iou(pred, gt, resolution=16)
        comp = compute_reconstruction_completeness(pred, gt, threshold=0.1)
        
        all_cd.extend(cd.tolist())
        all_iou.extend(iou.tolist())
        all_comp.extend(comp.tolist())

print('=== Evaluation Results ===')
print(f'Chamfer Distance : {np.mean(all_cd):.6f} ± {np.std(all_cd):.6f}')
print(f'IoU              : {np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}')
print(f'Completeness     : {np.mean(all_comp):.4f} ± {np.std(all_comp):.4f}')

# Metric distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, name, color in zip(axes,
    [all_cd, all_iou, all_comp],
    ['Chamfer Distance', 'IoU', 'Completeness'],
    ['#2878D2', '#DC503C', '#3CBE5A']):
    ax.hist(data, bins=20, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(np.mean(data), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {np.mean(data):.4f}')
    ax.set_title(name, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle('Metric Distributions on Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 🖼️ Visualization

Side-by-side comparison of input images, predicted 3D point clouds, and ground truth.

In [ ]:
from src.visualization.point_cloud_viz import PointCloudVisualizer

# Get raw samples (unnormalized images for display)
raw_ds = SimulatedShapeNetDataset(
    num_samples=100, num_points=512, image_size=128,
    categories=['cube', 'sphere', 'cylinder', 'cone', 'torus'],
    seed=99999,
)

model.eval()

for i in range(5):
    raw_sample = raw_ds[i]
    norm_sample = val_ds[i]
    
    with torch.no_grad():
        pred = model(norm_sample['image'].unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
    
    fig = PointCloudVisualizer.plot_comparison(
        image=raw_sample['image'].permute(1, 2, 0).numpy(),
        pred_points=pred,
        gt_points=raw_sample['point_cloud'].numpy(),
        category=raw_sample['category_name'],
    )
    plt.show()

## 6. 🔍 Explainability

Gradient saliency maps showing which image regions influence the 3D reconstruction.

In [ ]:
from src.explainability.feature_importance import GradientSaliency

saliency_analyzer = GradientSaliency(model, device=device)

for i in range(3):
    norm_sample = val_ds[i]
    raw_sample = raw_ds[i]
    
    sal_map = saliency_analyzer.compute(norm_sample['image'].unsqueeze(0))
    
    fig = saliency_analyzer.plot(
        raw_sample['image'].permute(1, 2, 0).numpy(),
        sal_map,
        title=f"Gradient Saliency — {raw_sample['category_name'].upper()}",
    )
    plt.show()

## 7. 🔬 Latent Space Analysis

Examine how the encoder maps different shape categories into the latent space.

In [ ]:
from sklearn.decomposition import PCA

model.eval()
latents = []
cats = []

with torch.no_grad():
    for i in range(min(200, len(val_ds))):
        sample = val_ds[i]
        raw = raw_ds[i]
        z = model.encode(sample['image'].unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        latents.append(z)
        cats.append(raw['category_name'])

latents = np.stack(latents)
pca = PCA(n_components=2)
z_2d = pca.fit_transform(latents)

fig, ax = plt.subplots(figsize=(8, 6))
for cat in ['cube', 'sphere', 'cylinder', 'cone', 'torus']:
    mask = [c == cat for c in cats]
    ax.scatter(z_2d[mask, 0], z_2d[mask, 1], label=cat, alpha=0.7, s=30,
              color=colors.get(cat, '#888888'))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('Latent Space (PCA)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Total variance explained: {sum(pca.explained_variance_ratio_):.1%}')

---

### Summary

This notebook demonstrated the full pipeline:
- **Dataset**: 5 shape categories, procedurally generated
- **Model**: ResNet-18 encoder → MLP decoder → 3D point cloud
- **Training**: Chamfer Distance loss with gradient clipping
- **Evaluation**: IoU, Chamfer Distance, and reconstruction completeness
- **Explainability**: Gradient saliency maps
- **Latent Space**: PCA visualization of learned embeddings

For full-scale training, use: `python train.py --config configs/default.yaml`